# RigGrade free GPU test, from Google Drive (not a grade)

Runs RigGrade's smoke test and preflight check on a free Colab GPU. The code
is read from the Drive folder `Adviser 1 Files/riggrade-src-*`, checked
against its manifest, and the two reports are written back to Drive in
`Coordinator and Adviser 1 Shared Files/riggrade-reports/`. Nothing to
download or upload by hand. No results are sent anywhere else.

**Before you start:** Runtime > Change runtime type > T4 GPU > Save. Then
Runtime > Run all. Approve the Google Drive access prompt when cell 2 asks.
Expect 45 to 60 minutes, most of it installing vLLM.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Read and verify the code from Drive
Refuses to continue if any file differs from the manifest.

In [ ]:
import glob, hashlib, json, os
from google.colab import drive
drive.mount('/content/drive')
found = sorted(glob.glob('/content/drive/MyDrive/Adviser 1 Files/riggrade-src-*/MANIFEST.json'))
assert found, 'no riggrade-src-* folder with MANIFEST.json in "Adviser 1 Files"'
SRC = os.path.dirname(found[-1])
man = json.load(open(found[-1]))
ROOT = '/content/riggrade'
bad = []
for e in man['files']:
    text = open(os.path.join(SRC, e['drive_name']), encoding='utf-8').read()
    text = text.rstrip('\n') + '\n'
    if hashlib.sha256(text.encode('utf-8')).hexdigest() != e['sha256']:
        bad.append(e['path'])
        continue
    dest = os.path.join(ROOT, e['path'])
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    open(dest, 'w', encoding='utf-8').write(text)
assert not bad, 'checksum mismatch, not running: %s' % bad
os.chdir(ROOT)
print('verified %d files from %s (commit %s)' % (len(man['files']), SRC, man['commit']))

## 3. Install the inference server
Often 5 to 15 minutes. If Colab asks to restart, accept, then run cell 2 again and continue from here.

In [ ]:
!pip install -q vllm

## 4. Smoke test (about 10 to 20 minutes)

In [ ]:
import os; os.chdir('/content/riggrade')
!python3 -m runner.smoke --minutes 2

## 5. Preflight check (about 1 minute)

In [ ]:
!python3 -m runner.preflight

## 6. Save both reports to Drive

## Record the versions used
vLLM and torch are installed unpinned on a free notebook, so the exact versions are saved with the reports.

In [ ]:
import json, platform, subprocess, sys
def _ver(pkg):
    try:
        from importlib.metadata import version
        return version(pkg)
    except Exception as exc:
        return 'not installed (%s)' % type(exc).__name__
versions = {'python': platform.python_version(),
            'vllm': _ver('vllm'), 'torch': _ver('torch'), 'transformers': _ver('transformers'),
            'nvidia_smi': subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version',
                                          '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()}
open('versions.json', 'w').write(json.dumps(versions, indent=1) + '\n')
print(json.dumps(versions, indent=1))

In [ ]:
import datetime, shutil
OUT = '/content/drive/MyDrive/Coordinator and Adviser 1 Shared Files/riggrade-reports'
os.makedirs(OUT, exist_ok=True)
stamp = datetime.datetime.utcnow().strftime('%Y%m%dT%H%MZ')
missing = []
for name in ('smoke-report.json', 'preflight-report.json', 'versions.json'):
    if os.path.exists(name):
        shutil.copy(name, os.path.join(OUT, '%s-%s' % (stamp, name)))
        print('saved', os.path.join(OUT, '%s-%s' % (stamp, name)))
    else:
        missing.append(name)
assert not missing, 'REPORT MISSING, run not complete: %s' % missing